This file will be for decideing which lr_scheduler to use

From previous files, the lr for head is 0.0017378008365631102 and lr range for whole model is ((1.9054607491852948e-06)/10, (1.9054607491852948e-06)/4)
The no. of epochs for head remains const. at 3 and no. of epochs for whole model is 8 as found in prev. file
The batch size found is 16 from prev. file, 
The weight decay value chosen is 1e-3
The optimizer chosen is RMSProp

I will be trying different lr schedulers 
they are StepLR, MultiStepLR, CosineAnnealingLR, CosineAnnealingWarmRestarts, OneCycleLR, SequentialLR, ReduceLROnPlateau

In [2]:
import pandas as pd
import regex as re
from fastai.vision.all import *
import torch

In [3]:
df = pd.read_csv(r'D:\Traffic\labels_processed.csv')

In [4]:
def label_function(dpath):
    class_name = re.findall(r'(\d+)_.*\.png$', dpath.name)
    class_id = int(class_name[0])
    return class_id

In [5]:
path = Path(r'D:\Traffic\traffic_Data_processed\DATA')

In [6]:
lr_head = 0.0017378008365631102
lr_whole_model = 1.9054607491852948e-06

In [7]:
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [8]:
from torch.utils.data import Dataset
from PIL import Image

In [9]:
class dset(Dataset):
    def __init__(self, image_paths, transform = None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert("RGB")
        label = label_function(Path(image_path))
        if self.transform:
            image = self.transform(image)
        return image, label

In [10]:
image_paths = list(path.rglob("*.png"))

In [11]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

dataset = dset(image_paths = image_paths, transform = transform)

In [12]:
from torch.utils.data import random_split

ts = int(0.75*len(dataset))
vs = len(dataset) - ts

generator = torch.Generator().manual_seed(42)
train_dataset, valid_dataset = random_split(dataset, [ts, vs], generator)

In [13]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False
)

In [14]:
import torchvision

In [15]:
num_classes = 55

In [16]:
model_6_1 = torchvision.models.resnet34(weights="DEFAULT")
model_6_1.fc = nn.Linear(
    model_6_1.fc.in_features,
    num_classes
)

In [16]:
for param in model_6_1.parameters():
    param.requires_grad = False

for param in model_6_1.fc.parameters():
    param.requires_grad = True

In [17]:
device = torch.device("cuda")

In [23]:
img = Image.open(Path(r"D:\Traffic\traffic_Data_processed\DATA\6\6_11.png"))
img = transform(img)
img = img.unsqueeze(0)

In [24]:
o = model_6_1(img)

In [28]:
o, o.max(1), o.size()

(tensor([[ 0.3726,  1.6705, -0.0122, -0.1869,  0.0379, -0.5265, -0.0313,  0.8715,
           0.0849, -0.2947,  0.7269,  0.1746,  0.8458, -1.1373, -0.1965,  0.3734,
          -0.1076,  0.0704, -0.0672,  0.5946, -0.8328,  0.0526,  0.8411, -0.7193,
           0.5533,  0.3957, -0.4446, -0.5337, -0.0546, -0.1441,  1.1590,  0.3902,
          -0.2693, -0.1255, -0.2121,  0.1177,  0.5071, -0.4616,  0.4376, -0.3798,
          -0.9716,  0.0498, -0.2771,  0.0391,  1.2073, -0.2769, -0.4342,  0.2622,
          -0.3338, -0.8969, -0.5339,  0.4015,  0.9284,  0.6896, -0.4836]],
        grad_fn=<AddmmBackward0>),
 torch.return_types.max(
 values=tensor([1.6705], grad_fn=<MaxBackward0>),
 indices=tensor([1])),
 torch.Size([1, 55]))

In [40]:
optimizer_1_1 = torch.optim.RMSprop(
    model_6_1.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

In [41]:
scheduler_1_1 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_1_1,
    max_lr=lr_head,
    epochs=3,
    steps_per_epoch=len(train_loader)
)

In [42]:
model_6_1 = model_6_1.to(device)

In [43]:
criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    model_6_1.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_1_1.zero_grad()

        outputs = model_6_1(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_1_1.step()

        scheduler_1_1.step()          

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total


    model_6_1.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_6_1(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/3 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/3 | Train Acc: 82.46% | Valid Acc: 89.98%
Epoch 2/3 | Train Acc: 89.62% | Valid Acc: 93.55%
Epoch 3/3 | Train Acc: 96.40% | Valid Acc: 97.50%


In [44]:
for param in model_6_1.parameters():
    param.requires_grad = True

In [45]:
param_groups = [
    {"params": model_6_1.conv1.parameters()},
    {"params": model_6_1.layer1.parameters()},
    {"params": model_6_1.layer2.parameters()},
    {"params": model_6_1.layer3.parameters()},
    {"params": model_6_1.layer4.parameters()},
    {"params": model_6_1.fc.parameters()},
]

low_lr = lr_whole_model / 10
high_lr = lr_whole_model / 4

max_lrs = torch.logspace(
    torch.log10(torch.tensor(low_lr)),
    torch.log10(torch.tensor(high_lr)),
    steps=len(param_groups)
).tolist()

In [46]:
optimizer_1_2 = torch.optim.RMSprop(
    param_groups,
    weight_decay=1e-3
)

In [47]:
scheduler_1_2 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_1_2,
    max_lr=max_lrs,
    epochs=8,
    steps_per_epoch=len(train_loader)
)

In [48]:
criterion = nn.CrossEntropyLoss()

for epoch in range(8):
    model_6_1.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_1_2.zero_grad()

        outputs = model_6_1(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_1_2.step()

        scheduler_1_2.step()          

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total


    model_6_1.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_6_1(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/8 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/8 | Train Acc: 99.07% | Valid Acc: 97.98%
Epoch 2/8 | Train Acc: 99.07% | Valid Acc: 98.36%
Epoch 3/8 | Train Acc: 99.52% | Valid Acc: 98.65%
Epoch 4/8 | Train Acc: 99.87% | Valid Acc: 98.65%
Epoch 5/8 | Train Acc: 99.90% | Valid Acc: 98.75%
Epoch 6/8 | Train Acc: 99.87% | Valid Acc: 98.84%
Epoch 7/8 | Train Acc: 99.87% | Valid Acc: 98.84%
Epoch 8/8 | Train Acc: 99.94% | Valid Acc: 98.84%


In [21]:
model_6_2 = torchvision.models.resnet34(weights="DEFAULT")
model_6_2.fc = nn.Linear(
    model_6_2.fc.in_features,
    num_classes
)

model_6_2 = model_6_2.to(device)

for param in model_6_2.parameters():
    param.requires_grad = False

for param in model_6_2.fc.parameters():
    param.requires_grad = True

optimizer_2_1 = torch.optim.RMSprop(
    model_6_2.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_2_1 = torch.optim.lr_scheduler.StepLR(
    optimizer_2_1,
    step_size = 2,
    gamma = 0.1
)

criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    model_6_2.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_2_1.zero_grad()

        outputs = model_6_2(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_2_1.step()          

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    scheduler_2_1.step()

    train_acc = 100 * train_correct / train_total


    model_6_2.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_6_2(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/3 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/3 | Train Acc: 67.72% | Valid Acc: 87.28%
Epoch 2/3 | Train Acc: 90.97% | Valid Acc: 88.05%
Epoch 3/3 | Train Acc: 96.63% | Valid Acc: 94.99%


In [22]:
for param in model_6_2.parameters():
    param.requires_grad = True

low_lr = lr_whole_model / 10
high_lr = lr_whole_model / 4

max_lrs = torch.logspace(
    torch.log10(torch.tensor(low_lr)),
    torch.log10(torch.tensor(high_lr)),
    steps=6
).tolist()

param_groups = [
    {"params": model_6_2.conv1.parameters(), "lr": max_lrs[0]},
    {"params": model_6_2.layer1.parameters(), "lr": max_lrs[1]},
    {"params": model_6_2.layer2.parameters(), "lr": max_lrs[2]},
    {"params": model_6_2.layer3.parameters(), "lr": max_lrs[3]},
    {"params": model_6_2.layer4.parameters(), "lr": max_lrs[4]},
    {"params": model_6_2.fc.parameters(), "lr": max_lrs[5]},
]

optimizer_2_2 = torch.optim.RMSprop(
    param_groups,
    weight_decay=1e-3
)

scheduler_2_2 = torch.optim.lr_scheduler.StepLR(
    optimizer_2_2,
    step_size = 2,
    gamma = 0.1
)

criterion = nn.CrossEntropyLoss()

for epoch in range(8):
    model_6_2.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_2_2.zero_grad()

        outputs = model_6_2(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_2_2.step()          

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    scheduler_2_2.step()

    train_acc = 100 * train_correct / train_total


    model_6_2.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_6_2(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/8 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/8 | Train Acc: 97.78% | Valid Acc: 95.38%
Epoch 2/8 | Train Acc: 98.20% | Valid Acc: 95.86%
Epoch 3/8 | Train Acc: 98.36% | Valid Acc: 95.66%
Epoch 4/8 | Train Acc: 98.46% | Valid Acc: 95.86%
Epoch 5/8 | Train Acc: 98.52% | Valid Acc: 95.76%
Epoch 6/8 | Train Acc: 98.27% | Valid Acc: 95.95%
Epoch 7/8 | Train Acc: 98.43% | Valid Acc: 95.95%
Epoch 8/8 | Train Acc: 98.27% | Valid Acc: 96.05%


In [23]:
model_6_3 = torchvision.models.resnet34(weights="DEFAULT")
model_6_3.fc = nn.Linear(
    model_6_3.fc.in_features,
    num_classes
)

model_6_3 = model_6_3.to(device)

for param in model_6_3.parameters():
    param.requires_grad = False

for param in model_6_3.fc.parameters():
    param.requires_grad = True

optimizer_3_1 = torch.optim.RMSprop(
    model_6_3.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_3_1 = torch.optim.lr_scheduler.MultiStepLR(
    optimizer_3_1,
    milestones = [2],
    gamma = 0.1
)

criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    model_6_3.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_3_1.zero_grad()

        outputs = model_6_3(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_3_1.step()          

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    scheduler_3_1.step()

    train_acc = 100 * train_correct / train_total


    model_6_3.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_6_3(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/3 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/3 | Train Acc: 66.91% | Valid Acc: 86.80%
Epoch 2/3 | Train Acc: 91.29% | Valid Acc: 89.40%
Epoch 3/3 | Train Acc: 96.27% | Valid Acc: 94.99%


In [24]:
for param in model_6_3.parameters():
    param.requires_grad = True

low_lr = lr_whole_model / 10
high_lr = lr_whole_model / 4

max_lrs = torch.logspace(
    torch.log10(torch.tensor(low_lr)),
    torch.log10(torch.tensor(high_lr)),
    steps=6
).tolist()

param_groups = [
    {"params": model_6_3.conv1.parameters(), "lr": max_lrs[0]},
    {"params": model_6_3.layer1.parameters(), "lr": max_lrs[1]},
    {"params": model_6_3.layer2.parameters(), "lr": max_lrs[2]},
    {"params": model_6_3.layer3.parameters(), "lr": max_lrs[3]},
    {"params": model_6_3.layer4.parameters(), "lr": max_lrs[4]},
    {"params": model_6_3.fc.parameters(), "lr": max_lrs[5]},
]

optimizer_3_2 = torch.optim.RMSprop(
    param_groups,
    weight_decay=1e-3
)

scheduler_3_2 = torch.optim.lr_scheduler.MultiStepLR(
    optimizer_3_2,
    milestones = [2, 5, 7],
    gamma = 0.1
)

criterion = nn.CrossEntropyLoss()

for epoch in range(8):
    model_6_3.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_3_2.zero_grad()

        outputs = model_6_3(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_3_2.step()          

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    scheduler_3_2.step()

    train_acc = 100 * train_correct / train_total


    model_6_3.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_6_3(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/8 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/8 | Train Acc: 97.65% | Valid Acc: 95.47%
Epoch 2/8 | Train Acc: 97.91% | Valid Acc: 95.76%
Epoch 3/8 | Train Acc: 98.43% | Valid Acc: 96.05%
Epoch 4/8 | Train Acc: 98.07% | Valid Acc: 95.76%
Epoch 5/8 | Train Acc: 98.30% | Valid Acc: 96.24%
Epoch 6/8 | Train Acc: 98.10% | Valid Acc: 96.24%
Epoch 7/8 | Train Acc: 98.14% | Valid Acc: 95.95%
Epoch 8/8 | Train Acc: 98.07% | Valid Acc: 95.76%


In [25]:
model_6_4 = torchvision.models.resnet34(weights="DEFAULT")
model_6_4.fc = nn.Linear(
    model_6_4.fc.in_features,
    num_classes
)

model_6_4 = model_6_4.to(device)

for param in model_6_4.parameters():
    param.requires_grad = False

for param in model_6_4.fc.parameters():
    param.requires_grad = True

optimizer_4_1 = torch.optim.RMSprop(
    model_6_4.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_4_1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_4_1,
    T_max = 3,
    eta_min = 1e-6
)

criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    model_6_4.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_4_1.zero_grad()

        outputs = model_6_4(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_4_1.step()          

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    scheduler_4_1.step()

    train_acc = 100 * train_correct / train_total


    model_6_4.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_6_4(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/3 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/3 | Train Acc: 66.08% | Valid Acc: 84.49%
Epoch 2/3 | Train Acc: 91.04% | Valid Acc: 90.37%
Epoch 3/3 | Train Acc: 95.66% | Valid Acc: 94.89%


In [26]:
for param in model_6_4.parameters():
    param.requires_grad = True

low_lr = lr_whole_model / 10
high_lr = lr_whole_model / 4

max_lrs = torch.logspace(
    torch.log10(torch.tensor(low_lr)),
    torch.log10(torch.tensor(high_lr)),
    steps=6
).tolist()

param_groups = [
    {"params": model_6_4.conv1.parameters(), "lr": max_lrs[0]},
    {"params": model_6_4.layer1.parameters(), "lr": max_lrs[1]},
    {"params": model_6_4.layer2.parameters(), "lr": max_lrs[2]},
    {"params": model_6_4.layer3.parameters(), "lr": max_lrs[3]},
    {"params": model_6_4.layer4.parameters(), "lr": max_lrs[4]},
    {"params": model_6_4.fc.parameters(), "lr": max_lrs[5]},
]

optimizer_4_2 = torch.optim.RMSprop(
    param_groups,
    weight_decay=1e-3
)

scheduler_4_2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_4_2,
    T_max = 8,
    eta_min = 1e-7
)

criterion = nn.CrossEntropyLoss()

for epoch in range(8):
    model_6_4.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_4_2.zero_grad()

        outputs = model_6_4(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_4_2.step()          

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    scheduler_4_2.step()

    train_acc = 100 * train_correct / train_total


    model_6_4.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_6_4(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/8 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/8 | Train Acc: 96.98% | Valid Acc: 95.86%
Epoch 2/8 | Train Acc: 97.56% | Valid Acc: 96.05%
Epoch 3/8 | Train Acc: 98.46% | Valid Acc: 96.24%
Epoch 4/8 | Train Acc: 98.33% | Valid Acc: 96.34%
Epoch 5/8 | Train Acc: 98.81% | Valid Acc: 96.72%
Epoch 6/8 | Train Acc: 98.46% | Valid Acc: 97.30%
Epoch 7/8 | Train Acc: 98.75% | Valid Acc: 97.01%
Epoch 8/8 | Train Acc: 98.78% | Valid Acc: 97.21%


In [27]:
model_6_5 = torchvision.models.resnet34(weights="DEFAULT")
model_6_5.fc = nn.Linear(
    model_6_5.fc.in_features,
    num_classes
)

model_6_5 = model_6_5.to(device)

for param in model_6_5.parameters():
    param.requires_grad = False

for param in model_6_5.fc.parameters():
    param.requires_grad = True

optimizer_5_1 = torch.optim.RMSprop(
    model_6_5.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_5_1 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_5_1,
    T_0=len(train_loader),
    T_mult = 1,
    eta_min = 1e-6
)

criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    model_6_5.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_5_1.zero_grad()

        outputs = model_6_5(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_5_1.step()   

        scheduler_5_1.step()       

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total


    model_6_5.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_6_5(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/3 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/3 | Train Acc: 69.51% | Valid Acc: 86.71%
Epoch 2/3 | Train Acc: 88.69% | Valid Acc: 91.62%
Epoch 3/3 | Train Acc: 91.68% | Valid Acc: 93.83%


In [28]:
for param in model_6_5.parameters():
    param.requires_grad = True

low_lr = lr_whole_model / 10
high_lr = lr_whole_model / 4

max_lrs = torch.logspace(
    torch.log10(torch.tensor(low_lr)),
    torch.log10(torch.tensor(high_lr)),
    steps=6
).tolist()

param_groups = [
    {"params": model_6_5.conv1.parameters(), "lr": max_lrs[0]},
    {"params": model_6_5.layer1.parameters(), "lr": max_lrs[1]},
    {"params": model_6_5.layer2.parameters(), "lr": max_lrs[2]},
    {"params": model_6_5.layer3.parameters(), "lr": max_lrs[3]},
    {"params": model_6_5.layer4.parameters(), "lr": max_lrs[4]},
    {"params": model_6_5.fc.parameters(), "lr": max_lrs[5]},
]

optimizer_5_2 = torch.optim.RMSprop(
    param_groups,
    weight_decay=1e-3
)

scheduler_5_2 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_5_2,
    T_0 = 2*len(train_loader),
    T_mult = 1,
    eta_min = 1e-7
)

criterion = nn.CrossEntropyLoss()

for epoch in range(8):
    model_6_5.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_5_2.zero_grad()

        outputs = model_6_5(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_5_2.step()

        scheduler_5_2.step()          

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)


    train_acc = 100 * train_correct / train_total


    model_6_5.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_6_5(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/8 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/8 | Train Acc: 96.98% | Valid Acc: 94.80%
Epoch 2/8 | Train Acc: 97.14% | Valid Acc: 94.22%
Epoch 3/8 | Train Acc: 97.21% | Valid Acc: 95.38%
Epoch 4/8 | Train Acc: 97.53% | Valid Acc: 95.47%
Epoch 5/8 | Train Acc: 97.78% | Valid Acc: 95.66%
Epoch 6/8 | Train Acc: 97.91% | Valid Acc: 96.05%
Epoch 7/8 | Train Acc: 97.91% | Valid Acc: 96.72%
Epoch 8/8 | Train Acc: 98.20% | Valid Acc: 96.92%


In [29]:
model_6_6 = torchvision.models.resnet34(weights="DEFAULT")
model_6_6.fc = nn.Linear(
    model_6_6.fc.in_features,
    num_classes
)

model_6_6 = model_6_6.to(device)

for param in model_6_6.parameters():
    param.requires_grad = False

for param in model_6_6.fc.parameters():
    param.requires_grad = True

optimizer_6_1 = torch.optim.RMSprop(
    model_6_6.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler1_6_1 = torch.optim.lr_scheduler.LinearLR(
    optimizer_6_1,
    start_factor = 0.1,
    end_factor = 1.0,
    total_iters = 1
)

scheduler2_6_1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_6_1,
    T_max = 2,
    eta_min = 1e-6
)

scheduler_6_1 = torch.optim.lr_scheduler.SequentialLR(
    optimizer_6_1,
    schedulers = [scheduler1_6_1, scheduler2_6_1],
    milestones = [1]
)

criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    model_6_6.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_6_1.zero_grad()

        outputs = model_6_6(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_6_1.step()          

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    scheduler_6_1.step()

    train_acc = 100 * train_correct / train_total


    model_6_6.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_6_6(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/3 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/3 | Train Acc: 39.13% | Valid Acc: 63.58%
Epoch 2/3 | Train Acc: 75.33% | Valid Acc: 87.09%
Epoch 3/3 | Train Acc: 92.80% | Valid Acc: 92.68%


In [30]:
for param in model_6_6.parameters():
    param.requires_grad = True

low_lr = lr_whole_model / 10
high_lr = lr_whole_model / 4

max_lrs = torch.logspace(
    torch.log10(torch.tensor(low_lr)),
    torch.log10(torch.tensor(high_lr)),
    steps=6
).tolist()

param_groups = [
    {"params": model_6_6.conv1.parameters(), "lr": max_lrs[0]},
    {"params": model_6_6.layer1.parameters(), "lr": max_lrs[1]},
    {"params": model_6_6.layer2.parameters(), "lr": max_lrs[2]},
    {"params": model_6_6.layer3.parameters(), "lr": max_lrs[3]},
    {"params": model_6_6.layer4.parameters(), "lr": max_lrs[4]},
    {"params": model_6_6.fc.parameters(), "lr": max_lrs[5]},
]

optimizer_6_2 = torch.optim.RMSprop(
    param_groups,
    weight_decay=1e-3
)

scheduler1_6_2 = torch.optim.lr_scheduler.LinearLR(
    optimizer_6_2,
    start_factor = 0.1,
    end_factor = 1.0,
    total_iters = 2
)

scheduler2_6_2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_6_2,
    T_max = 6,
    eta_min = 1e-7
)

scheduler_6_2 = torch.optim.lr_scheduler.SequentialLR(
    optimizer_6_2,
    schedulers = [scheduler1_6_2, scheduler2_6_2], 
    milestones = [2],
)

criterion = nn.CrossEntropyLoss()

for epoch in range(8):
    model_6_6.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_6_2.zero_grad()

        outputs = model_6_6(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_6_2.step()          

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    scheduler_6_2.step()

    train_acc = 100 * train_correct / train_total


    model_6_6.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_6_6(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/8 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/8 | Train Acc: 95.18% | Valid Acc: 92.87%
Epoch 2/8 | Train Acc: 95.41% | Valid Acc: 93.26%
Epoch 3/8 | Train Acc: 96.05% | Valid Acc: 94.41%
Epoch 4/8 | Train Acc: 96.92% | Valid Acc: 95.09%
Epoch 5/8 | Train Acc: 97.37% | Valid Acc: 95.38%
Epoch 6/8 | Train Acc: 97.62% | Valid Acc: 95.95%
Epoch 7/8 | Train Acc: 97.88% | Valid Acc: 96.24%
Epoch 8/8 | Train Acc: 98.07% | Valid Acc: 96.24%


In [31]:
model_6_7 = torchvision.models.resnet34(weights="DEFAULT")
model_6_7.fc = nn.Linear(
    model_6_7.fc.in_features,
    num_classes
)

model_6_7 = model_6_7.to(device)

for param in model_6_7.parameters():
    param.requires_grad = False

for param in model_6_7.fc.parameters():
    param.requires_grad = True

optimizer_7_1 = torch.optim.RMSprop(
    model_6_7.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_7_1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_7_1,
    mode = "max",
    factor = 0.1,
    patience = 1
)

criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    model_6_7.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_7_1.zero_grad()

        outputs = model_6_7(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_7_1.step()          

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total


    model_6_7.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_6_7(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    scheduler_7_1.step(valid_acc)

    print(
        f"Epoch {epoch+1}/3 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/3 | Train Acc: 67.17% | Valid Acc: 89.02%
Epoch 2/3 | Train Acc: 90.59% | Valid Acc: 90.94%
Epoch 3/3 | Train Acc: 94.31% | Valid Acc: 93.55%


In [32]:
for param in model_6_7.parameters():
    param.requires_grad = True

low_lr = lr_whole_model / 10
high_lr = lr_whole_model / 4

max_lrs = torch.logspace(
    torch.log10(torch.tensor(low_lr)),
    torch.log10(torch.tensor(high_lr)),
    steps=6
).tolist()

param_groups = [
    {"params": model_6_7.conv1.parameters(), "lr": max_lrs[0]},
    {"params": model_6_7.layer1.parameters(), "lr": max_lrs[1]},
    {"params": model_6_7.layer2.parameters(), "lr": max_lrs[2]},
    {"params": model_6_7.layer3.parameters(), "lr": max_lrs[3]},
    {"params": model_6_7.layer4.parameters(), "lr": max_lrs[4]},
    {"params": model_6_7.fc.parameters(), "lr": max_lrs[5]},
]

optimizer_7_2 = torch.optim.RMSprop(
    param_groups,
    weight_decay=1e-3
)

scheduler_7_2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_7_2,
    mode = "max",
    factor = 0.1,
    patience = 2
)

criterion = nn.CrossEntropyLoss()

for epoch in range(8):
    model_6_7.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_7_2.zero_grad()

        outputs = model_6_7(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_7_2.step()          

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total

    model_6_7.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_6_7(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    scheduler_7_2.step(valid_acc)

    print(
        f"Epoch {epoch+1}/8 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/8 | Train Acc: 96.82% | Valid Acc: 96.05%
Epoch 2/8 | Train Acc: 97.88% | Valid Acc: 96.53%
Epoch 3/8 | Train Acc: 98.46% | Valid Acc: 96.92%
Epoch 4/8 | Train Acc: 98.59% | Valid Acc: 97.21%
Epoch 5/8 | Train Acc: 99.07% | Valid Acc: 97.50%
Epoch 6/8 | Train Acc: 99.10% | Valid Acc: 97.69%
Epoch 7/8 | Train Acc: 99.26% | Valid Acc: 97.98%
Epoch 8/8 | Train Acc: 99.65% | Valid Acc: 98.17%


This means that OneCycleLR is perfect for our setup